# Leksjon 13 - Agentminne med Cognee Kunnskapsgrafer


## Oppsett

Denne notatboken viser hvordan man kan bygge en intelligent **kodeassistent** med vedvarende minne ved hjelp av [**Cognee**](https://www.cognee.ai/) kunnskapsgrafer og **Microsoft Agent Framework** (MAF).

Cognee omformer ustrukturert tekst til en strukturert, spørrbar kunnskapsgraf støttet av vektorinnbeddinger — som gir agenten din et rikt, relasjonsbevisst langtidsminne.

### Hva du vil lære
1. **Bygge kunnskapsgrafer**: Omforme utviklerprofiler og beste praksis til strukturert, spørrbar kunnskap.
2. **Integrere Cognee med MAF**: Bruke `@tool`-funksjoner for at en MAF-agent skal kunne spørre Cognees kunnskapsgraf.
3. **Sessionsbevisste samtaler**: Opprettholde kontekst over flere spørsmål i samme økt.
4. **Langtidsminne**: Bevare viktig kunnskap på tvers av økter og hente den frem i nye samtaler.

### Forutsetninger
- Python 3.9+
- Redis som kjører lokalt (`docker run -d -p 6379:6379 redis`) for øktbehandling
- En LLM API-nøkkel (f.eks. OpenAI) — sett `LLM_API_KEY` i `.env`
- `CACHING=true` i `.env` (påkrevd for Cognee-økter)
- Et Azure AI Foundry-prosjekt med en distribuert chatmodell
- `AZURE_AI_PROJECT_ENDPOINT` og `AZURE_AI_MODEL_DEPLOYMENT_NAME` i `.env`
- Azure CLI autentisert (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity "cognee[redis]==0.4.0" -q

In [ ]:
import os
from pathlib import Path
from typing import Annotated

from dotenv import load_dotenv

load_dotenv()

os.environ["LLM_API_KEY"] = os.getenv("LLM_API_KEY", "")
os.environ["CACHING"] = os.getenv("CACHING", "true")

import cognee
from cognee.modules.search.types import SearchType

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

print(f"Cognee version: {cognee.__version__}")
print(f"CACHING: {os.environ.get('CACHING')}")

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ AzureAIProjectAgentProvider created")

## Typer av Agent-minne

Denne notatboken utforsker de samme tre minnetypene fra hovednotatboken for leksjon 13, men bruker Cognee som backend for langtidsminnet:

| Minne Type | Mekanisme | Levretid |
|---|---|---|
| **Arbeids** | `agent.create_session()` (MAF) | Enkel samtale |
| **Korttids** | Cognee session cache (Redis) | Enkel økt |
| **Langtids** | Cognee knowledge graph + vektorer | Permanent |

### Cognees minnearkitektur
```
┌──────────────────────────┐
│      Raw Data            │  (developer profiles, docs, conversations)
└───────────┬──────────────┘
            │  cognee.add() + cognee.cognify()
            ▼
┌──────────────────────────────────────────┐
│  Knowledge Graph + Vector Embeddings     │
└───────────┬──────────────────────────────┘
            │  cognee.search()
            ▼
┌──────────────────┐       ┌────────────────┐
│  MAF Agent       │──────▶│  @tool funcs   │
│  (AgentSession)  │       │  wrapping       │
│                  │       │  cognee.search  │
└──────────────────┘       └────────────────┘
```


## Forbered Cognee-lagring


In [ ]:
DATA_ROOT = Path('.data_storage').resolve()
SYSTEM_ROOT = Path('.cognee_system').resolve()

DATA_ROOT.mkdir(parents=True, exist_ok=True)
SYSTEM_ROOT.mkdir(parents=True, exist_ok=True)

cognee.config.data_root_directory(str(DATA_ROOT))
cognee.config.system_root_directory(str(SYSTEM_ROOT))

await cognee.prune.prune_data()
await cognee.prune.prune_system(metadata=True)
print("✅ Cognee storage configured and reset")

## Del 1 — Bygge kunnskapsbasen

Vi tar inn tre typer data for å lage en omfattende kunnskapsbase for vår kodingassistent:

1. **Utviklerprofil** — personlig ekspertise og teknisk bakgrunn
2. **Pythons beste praksis** — Zen of Python med praktiske retningslinjer
3. **Historiske samtaler** — tidligere spørsmål og svar mellom utviklere og AI-assistenter


In [ ]:
developer_intro = (
    "Hi, I'm an AI/Backend engineer. "
    "I build FastAPI services with Pydantic, heavy asyncio/aiohttp pipelines, "
    "and production testing via pytest-asyncio. "
    "I've shipped low-latency APIs on AWS, Azure, and GoogleCloud."
)

python_zen_principles = """
# The Zen of Python: Practical Guide

## Key Principles With Guidance

### 1. Beautiful is better than ugly
Prefer descriptive names, clear structure, and consistent formatting.

### 2. Explicit is better than implicit
Be clear about behavior, imports, and types.

### 3. Simple is better than complex
Choose straightforward solutions first.

### 4. Flat is better than nested
Use early returns to reduce indentation.

## Modern Python Tie-ins
- Type hints reinforce explicitness
- Context managers enforce safe resource handling
- Dataclasses improve readability for data containers
"""

human_agent_conversations = """
"conversations": [
    {
      "topic": "async/await patterns",
      "user_query": "I'm building a web scraper that needs to handle thousands of URLs concurrently. What's the best way to structure this with asyncio?",
      "assistant_response": "Use asyncio with aiohttp, a semaphore to cap concurrency, TCPConnector for connection pooling, and context managers for session lifecycle."
    },
    {
      "topic": "dataclass vs pydantic",
      "user_query": "When should I use dataclasses vs Pydantic models?",
      "assistant_response": "For API input/output, prefer Pydantic: runtime validation, type coercion, JSON serialization. Integrates cleanly with FastAPI."
    },
    {
      "topic": "testing patterns",
      "user_query": "What's the best approach for pytest with async functions?",
      "assistant_response": "Use pytest-asyncio, async fixtures, and an isolated test database or mocks to reliably test async code."
    },
    {
      "topic": "error handling and logging",
      "user_query": "What's the best approach for production-ready error management?",
      "assistant_response": "Centralized error handling with custom exceptions, structured logging, and FastAPI middleware."
    }
  ]
"""

print("✅ Data sources prepared")

In [ ]:
await cognee.add(developer_intro, node_set=["developer_data"])
await cognee.add(human_agent_conversations, node_set=["developer_data"])
await cognee.add(python_zen_principles, node_set=["principles_data"])

await cognee.cognify()
print("✅ Knowledge graph built")

## Visualiser kunnskapsgrafen

Cognee kan gjengi en interaktiv HTML-visualisering av enhetene og relasjonene den har trukket ut.


In [ ]:
from cognee import visualize_graph

await visualize_graph('./cognee_graph.html')
print("📊 Graph saved to cognee_graph.html — open it in a browser to explore.")

## Forbedre minnet med Memify

`memify()` analyserer kunnskapsgrafen og genererer intelligente regler — identifiserer mønstre, beste praksis og relasjoner mellom konsepter.


In [ ]:
await cognee.memify()
print("✅ Memory enriched with memify")

## Del 2 — MAF-agent med Cognee-verktøy

Nå oppretter vi en MAF-agent som kan spørre Cognees kunnskapsgraf via `@tool`-funksjoner. Dette lar agenten utnytte den fulle kraften av grafbevisst semantisk søk samtidig som den opprettholder samtalekontekst gjennom økter.


In [ ]:
@tool(approval_mode="never_require")
async def search_knowledge(
    query: Annotated[str, "Natural-language question to search the knowledge graph"],
) -> str:
    """Search the Cognee knowledge graph for relevant developer knowledge, best practices, and past conversations."""
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
    )
    if not results:
        return "No relevant knowledge found."
    return str(results)


@tool(approval_mode="never_require")
async def search_principles(
    query: Annotated[str, "Question about Python principles or best practices"],
) -> str:
    """Search only the Python principles subset of the knowledge graph."""
    from cognee.modules.engine.models.node_set import NodeSet
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
        node_type=NodeSet,
        node_name=["principles_data"],
    )
    if not results:
        return "No relevant principles found."
    return str(results)


print("✅ Cognee tools defined: search_knowledge, search_principles")

In [ ]:
coding_agent = await provider.create_agent(
    name="CodingAssistant",
    instructions=(
        "You are an expert coding assistant with access to a knowledge graph "
        "containing developer profiles, Python best practices, and past conversations.\n\n"
        "WORKFLOW:\n"
        "1. Use search_knowledge() to find relevant information from the full knowledge graph.\n"
        "2. Use search_principles() when the question is specifically about Python best practices.\n"
        "3. Combine retrieved knowledge with your own expertise to give comprehensive answers.\n"
        "4. Reference the developer's known tech stack (FastAPI, asyncio, Pydantic) when relevant."
    ),
)

print("✅ CodingAssistant agent created")

## Arbeidsminne med økter

`AgentSession` (opprettet via `agent.create_session()`) gir arbeidsminne innen en økt. Agenten kan referere tilbake til tidligere meldinger samtidig som den spør Cognees langsiktige kunnskapsgraf.


In [ ]:
session = coding_agent.create_session()

response = await coding_agent.run(
    "How does my AsyncWebScraper implementation align with Python's design principles?",
    session=session,
)
print("🤖 Agent:", response)

In [ ]:
response = await coding_agent.run(
    "Based on what you just said, when should I pick dataclasses versus Pydantic for this work?",
    session=session,
)
print("🤖 Agent:", response)
print("\n💡 The agent combined working memory (previous answer) with Cognee's knowledge graph.")

## Ny økt — Langtidsminnet vedvarer

Å starte en ny økt tømmer arbeidsminnet, men Cognee-kunnskapsgrafen er fortsatt tilgjengelig. Agenten kan hente den samme langtidskunnskapen i en helt ny samtale.


In [ ]:
session_2 = coding_agent.create_session()

response = await coding_agent.run(
    "What logging guidance should I follow for incident reviews?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 New session, but the agent still has access to the full Cognee knowledge graph.")

In [ ]:
response = await coding_agent.run(
    "How should variables be named according to Python best practices?",
    session=session_2,
)
print("🤖 Agent:", response)

## Sammendrag

I denne notebooken bygde du en kodeassistent som kombinerer **MAFs arbeidsminne** (`agent.create_session()`) med **Cognees langsiktige kunnskapsgraf**.

### Hva du har lært
1. **Kunnskapsgrafkonstruksjon**: Cognee tar inn ustrukturert tekst og bygger en graf + vektorminne.
2. **Grafberikelse med memify**: Avledede fakta og rikere relasjoner på toppen av din eksisterende graf.
3. **MAF + Cognee integrasjon**: `@tool`-funksjoner lar en MAF-agent spørre Cognees graf på en naturlig måte.
4. **Arbeidsminne + langtidsminne**: `AgentSession` (via `agent.create_session()`) gir sesjonskontekst, mens Cognee gir vedvarende kunnskap.
5. **Filtrert søk med NodeSets**: Målrett spesifikke delmengder av kunnskapsgrafen (f.eks. kun prinsipper).

### Viktige poeng
- **Cognee** forvandler rå tekst til strukturert, relasjonsbevisst minne — kraftigere enn en flat vektorlager.
- **`@tool`-funksjoner** bygger bro mellom MAF-agenter og eksterne kunnskapssystemer på en ryddig måte.
- **`AgentSession`** (via `agent.create_session()`) holder per-samtalekontekst atskilt fra langvarig kunnskap.
- Den samme kunnskapsgrafen betjener flere sesjoner og agenter.

### Virkelige bruksområder
- **Utvikler-copiloter**: Kodegjennomgang, hendelsesanalyse, arkitektassistenter
- **Kundevendte copiloter**: Supportagenter over produktdokumentasjon, ofte stilte spørsmål og CRM-notater
- **Interne ekspert-copiloter**: Politikk-, juridiske- eller sikkerhetsassistenter som resonerer over retningslinjer
- **Enhetlige datalag**: Kombiner strukturert og ustrukturert data til én spørrbar graf

### Neste steg
- Eksperimenter med tidsbevissthet i Cognee
- Definer en OWL-ontologi for domenespesifikk grafkvalitet
- Legg til brukerfeedback-løkker for å forbedre gjenfinning over tid
- Skaler til multi-agent systemer som deler samme Cognee-minnelag


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Ansvarsfraskrivelse**:
Dette dokumentet er oversatt ved hjelp av AI-oversettelsestjenesten [Co-op Translator](https://github.com/Azure/co-op-translator). Selv om vi streber etter nøyaktighet, vennligst vær oppmerksom på at automatiserte oversettelser kan inneholde feil eller unøyaktigheter. Det opprinnelige dokumentet på originalspråket bør betraktes som den autoritative kilden. For kritisk informasjon anbefales profesjonell menneskelig oversettelse. Vi er ikke ansvarlige for misforståelser eller feiltolkninger som måtte oppstå ved bruk av denne oversettelsen.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
